# M5 EDA — Pharma-DemandForecast

Exploratory analysis of the M5 dataset before modelling.

In [ ]:
import sys
sys.path.insert(0, '..')

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

plt.rcParams['figure.dpi'] = 120
sns.set_theme(style='whitegrid')

DATA_PATH = '../data/processed/long.parquet'

## 1. Data Shapes & Overview

In [ ]:
df = pd.read_parquet(DATA_PATH)
print(f'Shape: {df.shape}')
print(f'Columns: {list(df.columns)}')
print(f'\nDate range: {df["date"].min()} — {df["date"].max()}')
print(f'Unique series (id): {df["id"].nunique():,}')
print(f'Unique days (d): {df["d"].nunique():,}')
print(f'\nMissing values:')
print(df.isnull().sum()[df.isnull().sum() > 0])

## 2. Zero-Ratio Analysis (Intermittency)

In [ ]:
zero_ratio = df.groupby('id', observed=True)['sales'].apply(lambda s: (s == 0).mean())

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].hist(zero_ratio, bins=50, edgecolor='k', alpha=0.8)
axes[0].set_xlabel('Zero-sales ratio')
axes[0].set_ylabel('Number of series')
axes[0].set_title('Distribution of zero-sales ratio across 30,490 series')

pct_intermittent = (zero_ratio > (1 - 1/1.32)).mean() * 100
axes[1].bar(['Non-intermittent', 'Intermittent (ADI>1.32)'],
            [100 - pct_intermittent, pct_intermittent], color=['steelblue', 'tomato'])
axes[1].set_ylabel('% of series')
axes[1].set_title(f'{pct_intermittent:.1f}% of series are intermittent')

plt.tight_layout()
plt.show()

print(f'Median zero-ratio: {zero_ratio.median():.3f}')
print(f'Series with >50% zeros: {(zero_ratio > 0.5).sum():,} ({(zero_ratio > 0.5).mean()*100:.1f}%)')

## 3. Sales Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

nonzero_sales = df.loc[df['sales'] > 0, 'sales']
axes[0].hist(np.log1p(nonzero_sales), bins=60, edgecolor='k', alpha=0.8)
axes[0].set_xlabel('log(1 + sales)')
axes[0].set_ylabel('Count')
axes[0].set_title('Non-zero sales distribution (log scale)')

cat_sales = df.groupby('cat_id', observed=True)['sales'].mean()
cat_sales.plot.bar(ax=axes[1], color='steelblue', edgecolor='k')
axes[1].set_xlabel('Category')
axes[1].set_ylabel('Mean daily sales per series')
axes[1].set_title('Average sales by category')
axes[1].tick_params(axis='x', rotation=0)

plt.tight_layout()
plt.show()

## 4. M5 Hierarchy Counts

In [ ]:
level_counts = {
    'Level 1 (Total)': 1,
    'Level 2 (State)': df['state_id'].nunique(),
    'Level 3 (Store)': df['store_id'].nunique(),
    'Level 4 (Category)': df['cat_id'].nunique(),
    'Level 5 (Department)': df['dept_id'].nunique(),
    'Level 6 (State × Cat)': df.groupby(['state_id','cat_id'], observed=True).ngroups,
    'Level 7 (State × Dept)': df.groupby(['state_id','dept_id'], observed=True).ngroups,
    'Level 8 (Store × Cat)': df.groupby(['store_id','cat_id'], observed=True).ngroups,
    'Level 9 (Store × Dept)': df.groupby(['store_id','dept_id'], observed=True).ngroups,
    'Level 10 (Item)': df['item_id'].nunique(),
    'Level 11 (State × Item)': df.groupby(['state_id','item_id'], observed=True).ngroups,
    'Level 12 (Store × Item)': df['id'].nunique(),
}
hierarchy_df = pd.DataFrame.from_dict(level_counts, orient='index', columns=['Series Count'])
print(hierarchy_df.to_string())
print(f'\nTotal unique series across all levels: {sum(level_counts.values()):,}')

## 5. Seasonality Patterns

In [ ]:
df['dayofweek'] = pd.to_datetime(df['date']).dt.dayofweek
df['month'] = pd.to_datetime(df['date']).dt.month

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

dow_sales = df.groupby('dayofweek')['sales'].mean()
dow_labels = ['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun']
axes[0].bar(dow_labels, dow_sales.values, color='steelblue', edgecolor='k')
axes[0].set_title('Average sales by day of week')
axes[0].set_ylabel('Mean daily sales')

month_sales = df.groupby('month')['sales'].mean()
axes[1].bar(range(1, 13), month_sales.values, color='coral', edgecolor='k')
axes[1].set_xticks(range(1, 13))
axes[1].set_xticklabels(['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec'], rotation=45)
axes[1].set_title('Average sales by month')
axes[1].set_ylabel('Mean daily sales')

plt.tight_layout()
plt.show()

## 6. Price Variability

In [ ]:
if 'sell_price' in df.columns:
    price_range = df.groupby('item_id', observed=True)['sell_price'].agg(['min','max'])
    price_range['range'] = price_range['max'] - price_range['min']

    fig, axes = plt.subplots(1, 2, figsize=(14, 4))
    axes[0].hist(df['sell_price'].dropna(), bins=60, edgecolor='k', alpha=0.8)
    axes[0].set_xlabel('Sell price ($)')
    axes[0].set_ylabel('Count')
    axes[0].set_title('Distribution of sell prices')

    axes[1].hist(price_range['range'], bins=50, edgecolor='k', alpha=0.8, color='tomato')
    axes[1].set_xlabel('Price range per item (max - min, $)')
    axes[1].set_ylabel('Number of items')
    axes[1].set_title('Item price range variability')

    plt.tight_layout()
    plt.show()
    print(f'Items with price changes: {(price_range["range"] > 0).sum():,} / {len(price_range):,}')
else:
    print('sell_price column not found in long.parquet — run preprocess.py first')

## 7. Event Analysis

In [ ]:
if 'event_name_1' in df.columns:
    df['has_event'] = df['event_name_1'].notna()
    event_sales = df.groupby('has_event')['sales'].mean()

    fig, ax = plt.subplots(figsize=(7, 4))
    ax.bar(['No Event', 'Event Day'], event_sales.values, color=['steelblue','orange'], edgecolor='k')
    ax.set_title('Average sales: event vs non-event days')
    ax.set_ylabel('Mean daily sales')
    lift = (event_sales[True] / event_sales[False] - 1) * 100
    ax.text(1, event_sales[True] * 0.5, f'+{lift:.1f}%', ha='center', fontsize=12, color='red')
    plt.tight_layout()
    plt.show()

    # Per event type
    print('\nTop events by mean sales lift:')
    event_type_sales = (
        df.dropna(subset=['event_name_1'])
        .groupby('event_name_1', observed=True)['sales'].mean()
        .sort_values(ascending=False)
        .head(10)
    )
    print(event_type_sales)

## 8. SNAP Analysis

In [ ]:
snap_cols = [c for c in df.columns if c.startswith('snap_')]
if snap_cols:
    fig, axes = plt.subplots(1, len(snap_cols), figsize=(5 * len(snap_cols), 4))
    if len(snap_cols) == 1:
        axes = [axes]
    for ax, col in zip(axes, snap_cols):
        state = col.replace('snap_', '')
        state_df = df[df['state_id'] == state]
        snap_sales = state_df.groupby(col)['sales'].mean()
        ax.bar(['Non-SNAP', 'SNAP'], snap_sales.values, color=['steelblue','green'], edgecolor='k')
        ax.set_title(f'{state}: SNAP vs non-SNAP days')
        ax.set_ylabel('Mean daily sales')
        if len(snap_sales) == 2:
            lift = (snap_sales.values[1] / snap_sales.values[0] - 1) * 100
            ax.text(1, snap_sales.values[1] * 0.5, f'+{lift:.1f}%', ha='center', fontsize=11)
    plt.tight_layout()
    plt.show()